### Setup
**Before you run anything:** set `SCRAG_STORE` to the store you may read and `OPENROUTER_API_KEY` to your own key, in your environment, before starting Jupyter. The next cell checks both.

In [7]:
import os
# set "SCRAG_STORE" before importing scrag, this is the command for Windows but you will 
# need to modify for your machine

os.environ["SCRAG_STORE"] = r"C:\Users\katyng\Downloads\students-v1\students-v1"   # set before importing scrag

if not os.environ.get("SCRAG_STORE"):
    raise SystemExit("Set SCRAG_STORE to your store folder, then restart the kernel. See INSTALL.md")

In [8]:
import sys
from pathlib import Path
import textwrap

os.environ["HF_HUB_OFFLINE"] = "1"                 # use the cached model only
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"   # no "Loading weights" bar

# this notebook lives in notebooks/, so put the repository root on the import path
ROOT = Path.cwd() / "rag-development"
sys.path.insert(0, str(ROOT))

print("cwd:", Path.cwd())
print("ROOT:", ROOT)
print("scrag exists:", (ROOT / "scrag").is_dir())

from scrag.agents.ask import ask, show, show_prompt, CONTRACT

print("repo  :", ROOT)
print("store :", os.environ.get("SCRAG_STORE", "not set (the code will look in data/)"))
print("key   :", "set" if os.environ.get("OPENROUTER_API_KEY") else "NOT SET - the answer step will fail")

cwd: c:\Users\katyng\Downloads\AI_Agents
ROOT: c:\Users\katyng\Downloads\AI_Agents\rag-development
scrag exists: True
repo  : c:\Users\katyng\Downloads\AI_Agents\rag-development
store : C:\Users\katyng\Downloads\students-v1\students-v1
key   : set


# Asking the literature store a question

The store holds papers cut into chunks. This notebook asks it a question, has an LLM model write an answer, and then checks for the existence of every citation in that answer against the store.

| Step | Function |
| :--- | :--- |
| retrieve the chunks | Deterministic code: SQLite keyword search coupled with numpy dot product |
| write the answer | LLM model from prompt containing: contract, question and chunks |
| check existence of citation | Deterministic code: string comparison against the store |

Three steps:

1. **Retrieve.** Keyword search and meaning search over the store, their rankings merged. Returns text with metadata, never an answer.
2. **Answer.** The model is given the question, the chunks, and the citation contract (`scrag/agents/contract.md`). It may cite only what it was given.
3. **Gate.** Every citation is checked: does the paper exist in the store, was it actually retrieved for this question, and do any quoted words really occur in it. The gate reports; it never edits.

A citation that passes has been verified against the text. A citation that fails is shown with the reason.

Note: The model can read a real chunk, summarize it wrongly, attach the correct citation, and pass the gate cleanly. This is not a semantic check.

## 1. One question

`ask()` does all three steps (retrieve, answer and gate) and returns the result. 

k=n sets the number of chunks that are fetched and handed to the model, at most two per paper.

`show()` displays the result: the answer,  the papers-cited table, the gate's verdict on each citation, then the label mix.

The first call loads the embedding model and takes about ten seconds. Later calls in the same kernel are quick.

In [8]:
result = ask("can you list the best accepted markers of glial senescence in flies?", k=20) 
show(result) 

**can you list the best accepted markers of glial senescence in flies?**

The best-accepted markers of glial senescence in flies centre on AP1 transcriptional activity, detected with a fluorescent TRE-dsRed reporter line, which increases with age in a stereotyped and regionalized manner in the Drosophila brain (10.1038/s41514-024-00181-1#19). Alongside elevated AP1 activity, senescent glia in aged flies show increased senescence-associated β-galactosidase (SA-β-Gal) activity and accumulation of the DNA damage marker γH2Av (10.1038/s41514-024-00181-1#19). Astrocyte-like glial cells in Drosophila also upregulate these senescence-associated markers during normal aging, with AP1 and increased β-galactosidase activity being the most traditionally described (10.1186/s13024-025-00810-7#13). Lipid droplet accumulation, detected by BODIPY staining for neutral lipids, is an additional functional hallmark associated with AP1-positive senescent glia, increasing in size and number with age in the fly brain (10.1038/s41586-024-07516-8#12). Intermittent blockade of glial AP1 activity reduces AP1-target genes and senescence biomarkers without altering markers of mitochondrial status, confirming that AP1 activity, SA-β-Gal, and lipid droplet burden together constitute the core senescence signature in fly glia (10.1093/lifemeta/loae031#2).

<sub>anthropic/claude-sonnet-4.6, $0.0383; store C:\Users\katyng\Downloads\students-v1\students-v1; run C:\Users\katyng\Downloads\AI_Agents\rag-development\reports\runs\ask\2026-10-05_180304; gate pass</sub>

**Papers cited**

,citations,paper,kind,tier,doi
0,2,"Mansfield 2024, npj Aging",review,1,10.1038/s41514-024-00181-1
1,1,"Gildea 2025, Molecular Neurodegeneration",review,2,10.1186/s13024-025-00810-7
2,1,"Byrns 2024, Nature",research-article,core,10.1038/s41586-024-07516-8
3,1,"Reyes 2024, Life Metabolism",research-article,1,10.1093/lifemeta/loae031


**Every citation, as the gate judged it**

,citation,verdict,caveat,reason
0,10.1038/s41514-024-00181-1#19,pass,,
1,10.1038/s41514-024-00181-1#19,pass,,
2,10.1186/s13024-025-00810-7#13,pass,,
3,10.1038/s41586-024-07516-8#12,pass,,
4,10.1093/lifemeta/loae031#2,pass,,


**Label mix**: of the chunks retrieval offered, which kinds the answer actually used

retrieved  cited
category    value                             
tier        core                      2      1
            1                         5      2
            2                        13      1
kind        research-article         13      2
            review                    7      2
reliability xml text                 16      4
            pdf text                  2      0
            preprint                  2      0
store       baseline                 20      4

A dictionary is returned with 4 items:
1. the answer
2. list of the papers that were cited
3. whether the citation exists (the gate)
4. details about the retrieved papers and their usage in citations

|Tier|
|-----------------------------------------------------------|
| core: collaborator's initial list of papers               |
| tier 1: linked to two or more of the core papers          |
| tier 2: linked to one plus exceeds a similarity threshold |

Similarity threshold: Each candidate papers's abstract is embedded with the same model and revision the store's chunks use ( bge-small-en-v1.5), unit normalised. Each of the 25 core papers' abstracts are embedded the same way. The candidate's score is the maximum cosine against any single core abstract, not the mean, and the nearest core paper is recorded with it. So a paper close to one core paper scores high even if it is unrelated to the other 24, which is deliberate: the core set spans several topics.

The threshold value is 0.74, and it was calibrated. 150 single-link candidates near the boundary were labelled relevant or not by a person, 80 relevant and 70 not, and the threshold was set by the rule "the lowest similarity that keeps 90 percent of the relevant ones".


| Threshold | Admits | Precision | Recall |
| :--- | :--- | :--- | :--- |
| **0.74, chosen** | 96 | 0.75 | 0.90 |
| **0.76** | 77 | 0.84 | 0.81 |
| **0.80** | 49 | 0.96 | 0.59 |


The reasoning for favouring recall: in ranked retrieval an off-topic paper only surfaces for an off-topic question, whereas a relevant paper left out can never be cited.


Abstracts come from OpenAlex, falling back to Europe PMC, with the source recorded. A candidate with no abstract anywhere is unscored and can only enter through tier 1.

#### View the Contract
This is the suffix for the LLM query.

In [9]:
# Loop through each line and wrap it to your desired width (e.g., 80 characters)
for line in CONTRACT.splitlines():
    print(textwrap.fill(line, width=80))

You answer a question about the biology of ageing, senescence and
neurodegeneration from the literature store only. The message gives you the
question and, as JSON, the chunks the store's retrieval returned for it. Nothing
outside those chunks may be cited.

Rules:

1. Cite by copying a chunk's `cite` field exactly as given, for example
(10.1038/s41586-024-07516-8#3) or (10.1038/s41586-024-07516-8#0; abstract only).
Never shorten it, never drop or reword the part after the semicolon: it is a
caveat about the text you read and the reader needs it.
2. One citation per pair of brackets. To cite two chunks, write them as two
separate bracketed citations, (a; caveat) (b; caveat), never bundled inside one
pair of brackets.
3. Every sentence that states a finding ends with the citation of the chunk it
came from. A sentence with no citation states nothing from the literature.
4. Quotation marks are reserved for words copied character for character from a
chunk's `text` field. Inside a quotatio

### To Do
There are several things you can do to improve this response. These include:
1. rephrase the question
2. rephrase the contract
3. follow the suggestions in Ng's course like incorporating reflection
4. add external information to the prompt in addition to the contract, question and chunks. Ultimately this will include the data from the deterministic data analysis side. 
5. Change what is retrieved, not just what is written. There are a variety of ways to do this, but one thing is to create a more complex and complete retrieval query by asking an LLM model for candidate genes, markers and reporters relevant to the question. 

Play with this to get a sense of how things work. A simple starting point is just to get the response framed as a list, which is more useful for figuring out if it returned all the important measures of senescence.  

### My Experiments Start

In [11]:
import runpy

comparison = runpy.run_path(str(ROOT.parent / "model_comparison_experiments.py"))
configure = comparison['configure']
run_model_comparison = comparison['run_model_comparison']
score_outputs = comparison['score_outputs']
show_summary = comparison['show_summary']
make_semantic_audit_template = comparison['make_semantic_audit_template']
summarize_completed_audit = comparison['summarize_completed_audit']
MODELS = comparison['MODELS']
QUESTIONS = comparison['QUESTIONS']

# One repeat is a cost-controlled pilot. Change to repeats=3 after inspection.
CONFIG = configure(ROOT, models=MODELS, questions=QUESTIONS, k=20, repeats=1, corpus='hypothesis')
print('models:', CONFIG['models'])
print('questions:', [item['id'] for item in CONFIG['questions']])
print('run folder:', CONFIG['run_dir'])

models: {'Haiku 4.5': 'anthropic/claude-haiku-4.5', 'Sonnet 4.6': 'anthropic/claude-sonnet-4.6', 'Opus 4.6': 'anthropic/claude-opus-4.6', 'Fable 5.1': 'anthropic/claude-fable-5.1'}
questions: ['evidence-map', 'state-discrimination', 'causality', 'false-premise']
run folder: c:\Users\katyng\Downloads\AI_Agents\rag-development\reports\runs\model_comparison\2026-10-05_180734


In [12]:
# Paid API calls: four models times four questions times repeats.
comparison_results = run_model_comparison(CONFIG)
scored_results = score_outputs(comparison_results)
summary = show_summary(scored_results)

evidence-map 1 Haiku 4.5 GATE FAIL
evidence-map 1 Sonnet 4.6 pass
evidence-map 1 Opus 4.6 pass
evidence-map 1 Fable 5.1 ERROR
state-discrimination 1 Haiku 4.5 GATE FAIL
state-discrimination 1 Sonnet 4.6 GATE FAIL
state-discrimination 1 Opus 4.6 pass
state-discrimination 1 Fable 5.1 ERROR
causality 1 Haiku 4.5 GATE FAIL
causality 1 Sonnet 4.6 pass
causality 1 Opus 4.6 pass
causality 1 Fable 5.1 ERROR
false-premise 1 Haiku 4.5 GATE FAIL
false-premise 1 Sonnet 4.6 pass
false-premise 1 Opus 4.6 GATE FAIL
false-premise 1 Fable 5.1 ERROR


### Experiment 1: controlled complex-RAG comparison

Tests whether output length and citation or contract failures differ by model when the evidence, question, contract, temperature, and k are held fixed.

,model_label,runs,mean_words,median_words,mean_sentences,gate_pass_rate,contract_proxy_pass_rate,mean_invalid_citations,total_cost_usd
0,Haiku 4.5,4,181.00,185.0,5.00,0.00,0.00,1.75,0.054
2,Sonnet 4.6,4,253.25,242.5,4.75,0.75,0.50,0.50,0.171
1,Opus 4.6,4,271.50,271.0,5.50,0.75,0.75,0.25,0.290


,question_id,question_kind,repeat,model_label,retrieval_hash,word_count,sentence_count,citation_count,invalid_citation_count,citation_gate_pass,contract_proxy_pass,cost_usd,answer_path
8,causality,complex synthesis,1,Haiku 4.5,683a5117ac88ca7fe1570e62b680e18f8baaa3268df9c3...,191,5,6,2,False,False,0.013924,c:\Users\katyng\Downloads\AI_Agents\rag-develo...
10,causality,complex synthesis,1,Opus 4.6,683a5117ac88ca7fe1570e62b680e18f8baaa3268df9c3...,241,5,8,0,True,True,0.072025,c:\Users\katyng\Downloads\AI_Agents\rag-develo...
9,causality,complex synthesis,1,Sonnet 4.6,683a5117ac88ca7fe1570e62b680e18f8baaa3268df9c3...,238,5,7,0,True,True,0.042810,c:\Users\katyng\Downloads\AI_Agents\rag-develo...
0,evidence-map,complex synthesis,1,Haiku 4.5,ef2c320d8100cfe6924855da0c9d45c32ab7bd56662ce7...,179,5,5,2,False,False,0.014396,c:\Users\katyng\Downloads\AI_Agents\rag-develo...
2,evidence-map,complex synthesis,1,Opus 4.6,ef2c320d8100cfe6924855da0c9d45c32ab7bd56662ce7...,286,5,6,0,True,True,0.076860,c:\Users\katyng\Downloads\AI_Agents\rag-develo...
1,evidence-map,complex synthesis,1,Sonnet 4.6,ef2c320d8100cfe6924855da0c9d45c32ab7bd56662ce7...,247,5,7,0,True,True,0.045411,c:\Users\katyng\Downloads\AI_Agents\rag-develo...
12,false-premise,abstention stress test,1,Haiku 4.5,9408b99e438e396d1fc733163e34ef16ea574ad3e5a435...,133,4,3,2,False,False,0.013251,c:\Users\katyng\Downloads\AI_Agents\rag-develo...
14,false-premise,abstention stress test,1,Opus 4.6,9408b99e438e396d1fc733163e34ef16ea574ad3e5a435...,256,7,5,1,False,False,0.072185,c:\Users\katyng\Downloads\AI_Agents\rag-develo...
13,false-premise,abstention stress test,1,Sonnet 4.6,9408b99e438e396d1fc733163e34ef16ea574ad3e5a435...,124,3,2,0,True,False,0.039156,c:\Users\katyng\Downloads\AI_Agents\rag-develo...
4,state-discrimination,complex synthesis,1,Haiku 4.5,ef5fe2c22c68d4d5f0b27ee38de0839eaa422b9c64faa3...,221,6,4,1,False,False,0.012790,c:\Users\katyng\Downloads\AI_Agents\rag-develo...


In [13]:
# Automatic citation checks cannot establish semantic support.
audit_template = make_semantic_audit_template(scored_results)

# After a blinded reviewer completes the CSV in the run folder:
# semantic_summary = summarize_completed_audit(scored_results)

### Experiment 2: sentence-level semantic grounding audit

Mark literature-finding sentences and whether the saved retrieved chunks support them. This measures hallucination or overstatement that a syntactically valid citation gate cannot detect. Save the completed file as semantic_audit_completed.csv in the same folder.

,question_id,model_label,repeat,sentence_number,sentence,has_citation,cited_identifiers,citation_gate_pass_for_answer,is_literature_finding,support,auditor_notes,records_path,answer_path
0,evidence-map,Haiku 4.5,1,1,The retrieved literature provides direct evide...,False,,False,,,,c:\Users\katyng\Downloads\AI_Agents\rag-develo...,c:\Users\katyng\Downloads\AI_Agents\rag-develo...
1,evidence-map,Haiku 4.5,1,2,"In the Drosophila model system, senescent astr...",True,10.1186/s13024-025-00810-7#13,False,,,,c:\Users\katyng\Downloads\AI_Agents\rag-develo...,c:\Users\katyng\Downloads\AI_Agents\rag-develo...
2,evidence-map,Haiku 4.5,1,3,A direct mechanistic observation in aged Droso...,True,10.1038/s41586-024-07516-8#6; 10.1038/s41586-0...,False,,,,c:\Users\katyng\Downloads\AI_Agents\rag-develo...,c:\Users\katyng\Downloads\AI_Agents\rag-develo...
3,evidence-map,Haiku 4.5,1,4,The proposed mechanism involves aged neurons r...,True,10.1038/s41586-024-07516-8#16,False,,,,c:\Users\katyng\Downloads\AI_Agents\rag-develo...,c:\Users\katyng\Downloads\AI_Agents\rag-develo...
4,evidence-map,Haiku 4.5,1,5,A functional readout shows that mildly dampeni...,True,10.1038/s41586-024-07516-8#16,False,,,,c:\Users\katyng\Downloads\AI_Agents\rag-develo...,c:\Users\katyng\Downloads\AI_Agents\rag-develo...
5,evidence-map,Sonnet 4.6,1,1,"In the aged *Drosophila* brain, naturally occu...",True,10.1038/s41514-024-00181-1#19; 10.1038/s41586-...,True,,,,c:\Users\katyng\Downloads\AI_Agents\rag-develo...,c:\Users\katyng\Downloads\AI_Agents\rag-develo...
6,evidence-map,Sonnet 4.6,1,2,RNA-seq of FACS-isolated aged *Drosophila* neu...,True,10.1038/s41586-024-07516-8#6; 10.1038/s41586-0...,True,,,,c:\Users\katyng\Downloads\AI_Agents\rag-develo...,c:\Users\katyng\Downloads\AI_Agents\rag-develo...
7,evidence-map,Sonnet 4.6,1,3,"As a direct functional readout, mildly dampeni...",True,10.1038/s41586-024-07516-8#16,True,,,,c:\Users\katyng\Downloads\AI_Agents\rag-develo...,c:\Users\katyng\Downloads\AI_Agents\rag-develo...
8,evidence-map,Sonnet 4.6,1,4,The mechanism by which neuronal deterioration ...,True,10.1038/s41586-024-07516-8#16,True,,,,c:\Users\katyng\Downloads\AI_Agents\rag-develo...,c:\Users\katyng\Downloads\AI_Agents\rag-develo...
9,evidence-map,Sonnet 4.6,1,5,"Separately, reviews note that lipid-accumulati...",True,10.1186/s13024-025-00810-7#13,True,,,,c:\Users\katyng\Downloads\AI_Agents\rag-develo...,c:\Users\katyng\Downloads\AI_Agents\rag-develo...


c:\Users\katyng\Downloads\AI_Agents\rag-development\reports\runs\model_comparison\2026-10-05_180734\semantic_audit_template.csv


## Controlled numerical-evidence experiment (synthetic A/B inputs)

This section tests whether the answer changes correctly when a significant synthetic gene log2 fold change reverses sign while the question and literature stay fixed. The default design has **two questions ? two dataset versions ? three models ? two repetitions = 24 calls**. Set `NUM_REPEATS = 1` in the dry-run cell for a **12-call, single-repetition option**. These datasets are synthetic inputs, not findings from the cited papers.

For each question, the helper retrieves 20 candidates from the existing `students-v1` store once, selects four or five relevant chunks, and freezes their exact text and order. The full candidate retrieval and a human-reviewed file of narrowly supported literature claims are saved outside the model prompt. Only the selected chunks are available to the citation gate. The model sees a concise question, synthetic rows, the original citation contract, and a short dataset-citation extension. The five-sentence limit and prohibition on unsupported causal or gene-function claims remain in force.

The next cell runs **dry validation only**. It saves the complete execution order and a preflight table with estimated input and output tokens by model. Cost is left unavailable because this notebook has no verified pricing source. Estimates are based on prior recorded usage where available and otherwise on a labelled fallback; the 768-token output cap is also shown separately. The dry run makes no paid model calls.


In [10]:
import runpy

numerical = runpy.run_path(str(ROOT.parent / "numerical_evidence_experiment.py"))
num_configure = numerical["configure"]
num_run = numerical["run_experiment"]

NUM_REPEATS = 2  # Default: 24 calls. Set to 1 for the 12-call option.
NUM_DRY_CONFIG = num_configure(ROOT, repeats=NUM_REPEATS, max_tokens=768)
num_dry = num_run(NUM_DRY_CONFIG, dry_run=True)
print("Dry-run artifacts:", num_dry["run_dir"])
print("Planned calls:", len(num_dry["matrix"]))


Planned model calls: 24. Estimated usage by model (cost unavailable):


,model_label,model_id,calls,estimated_input_tokens_per_call_min,estimated_input_tokens_per_call_max,estimated_input_tokens_total,estimated_output_tokens_per_call,estimated_output_tokens_total,output_token_cap_per_call,output_token_cap_total,estimate_basis,cost_estimate_usd
0,Haiku 4.5,anthropic/claude-haiku-4.5,8,2830,3272,24408,217,1736,768,6144,median prior observed model tokens/character,None
1,Sonnet 4.6,anthropic/claude-sonnet-4.6,8,2677,3094,23084,250,2000,768,6144,fallback 0.28 tokens/character,None
2,Opus 4.6,anthropic/claude-opus-4.6,8,2796,3232,24112,335,2680,768,6144,median prior observed model tokens/character,None


Dry run validated 24 planned calls, two one-time candidate retrievals, curated frozen evidence, claims, paired inputs, and scoring fixtures. No model calls. c:\Users\katyng\Downloads\AI_Agents\rag-development\reports\runs\numerical_evidence\20261020T203131981931Z-b65977a6
Dry-run artifacts: c:\Users\katyng\Downloads\AI_Agents\rag-development\reports\runs\numerical_evidence\20261020T203131981931Z-b65977a6
Planned calls: 24


### Paid run (launch manually after reviewing the dry run)

Inspect the dry-run folder's `preflight_usage.csv`, `frozen_evidence/*/reviewed_literature_claims.json`, `execution_order.csv`, and synthetic datasets first. The paid cell below creates a **new** run using those same frozen candidates, prints its planned count and token estimates **before** any paid request, then sends only the first sequential batch of six. The following resume cell sends one more batch each time it is run. Requests use temperature zero and an initial 768-token output cap. A credit, key-limit, or connection failure stops a batch.

Every response, request, retry, finish reason, token usage, and available actual cost is saved. Responses that reach the cap or have a length finish reason are flagged as potentially truncated and excluded from automatic format-rate denominators. Numeric checks and citation-existence checks do not establish semantic support. Literature support, unsupported mechanisms, causal claims, and full A/B interpretation changes remain for human review in the exported review sheets. Earlier experiments remain separate.


In [11]:
# PAID, RUNS THE FIRST BATCH OF SIX CALLS.
NUM_PAID_CONFIG = num_configure(ROOT, repeats=NUM_REPEATS, max_tokens=768,
                                frozen_from=num_dry["run_dir"])
num_paid = num_run(NUM_PAID_CONFIG, dry_run=False)
print("Paid experiment folder:", num_paid["run_dir"])


Planned model calls: 24. Estimated usage by model (cost unavailable):


,model_label,model_id,calls,estimated_input_tokens_per_call_min,estimated_input_tokens_per_call_max,estimated_input_tokens_total,estimated_output_tokens_per_call,estimated_output_tokens_total,output_token_cap_per_call,output_token_cap_total,estimate_basis,cost_estimate_usd
0,Haiku 4.5,anthropic/claude-haiku-4.5,8,2830,3272,24408,217,1736,768,6144,median prior observed model tokens/character,None
1,Sonnet 4.6,anthropic/claude-sonnet-4.6,8,2677,3094,23084,250,2000,768,6144,fallback 0.28 tokens/character,None
2,Opus 4.6,anthropic/claude-opus-4.6,8,2796,3232,24112,335,2680,768,6144,median prior observed model tokens/character,None


1 Sonnet 4.6 relish B 2 success
2 Opus 4.6 atg8a A 1 success
3 Opus 4.6 atg8a B 2 success
4 Haiku 4.5 atg8a A 2 success
5 Haiku 4.5 atg8a A 1 success
6 Haiku 4.5 relish B 2 success
Batch attempted 6; 6/24 completed successfully. c:\Users\katyng\Downloads\AI_Agents\rag-development\reports\runs\numerical_evidence\20261020T203137237244Z-c1b7697d


,model_label,successful_calls,failed_calls,processing_failures,model_mismatches,potentially_truncated_calls,format_graded_calls,data_value_rate,data_p_rate,data_direction_rate,...,literature_citation_present_rate,five_sentence_rate,plain_prose_rate,paired_direction_rate,paired_conditions,mean_distinct_answers_per_repeated_condition,semantic_support,causal_claim_review,full_interpretation_change,semantic_repeat_consistency
0,Haiku 4.5,3,0,0,0,0,3,None,None,None,...,1.0,1.0,1.0,None,0,1.0,None,None,None,None
1,Opus 4.6,2,0,0,0,0,2,None,None,None,...,1.0,1.0,1.0,None,0,NaN,None,None,None,None
2,Sonnet 4.6,1,0,0,0,0,1,None,None,None,...,1.0,1.0,1.0,None,0,NaN,None,None,None,None


Paid experiment folder: c:\Users\katyng\Downloads\AI_Agents\rag-development\reports\runs\numerical_evidence\20261020T203137237244Z-c1b7697d


In [12]:
# PAID, RUN THIS CELL FOR EACH ADDITIONAL BATCH OF SIX, UNTIL YOU'VE DONE 24 SUCCESSFUL CALLS.
# each execution resumes one batch of up to six sequential calls.
# After a kernel restart, load the helper and use numerical["resume_config"](saved_run_dir).
num_paid = num_run(NUM_PAID_CONFIG, dry_run=False, resume=True)
print("Paid experiment folder:", num_paid["run_dir"])


Planned model calls: 24. Estimated usage by model (cost unavailable):


,model_label,model_id,calls,estimated_input_tokens_per_call_min,estimated_input_tokens_per_call_max,estimated_input_tokens_total,estimated_output_tokens_per_call,estimated_output_tokens_total,output_token_cap_per_call,output_token_cap_total,estimate_basis,cost_estimate_usd
0,Haiku 4.5,anthropic/claude-haiku-4.5,8,2830,3272,24408,217,1736,768,6144,median prior observed model tokens/character,NaN
1,Sonnet 4.6,anthropic/claude-sonnet-4.6,8,2677,3094,23084,250,2000,768,6144,fallback 0.28 tokens/character,NaN
2,Opus 4.6,anthropic/claude-opus-4.6,8,2796,3232,24112,335,2680,768,6144,median prior observed model tokens/character,NaN


7 Opus 4.6 atg8a A 2 api_failed
Stopped batch after an access, credit, or connection failure; inspect the saved attempt before resuming.
Batch attempted 1; 6/24 completed successfully. c:\Users\katyng\Downloads\AI_Agents\rag-development\reports\runs\numerical_evidence\20261020T203137237244Z-c1b7697d


,model_label,successful_calls,failed_calls,processing_failures,model_mismatches,potentially_truncated_calls,format_graded_calls,data_value_rate,data_p_rate,data_direction_rate,...,literature_citation_present_rate,five_sentence_rate,plain_prose_rate,paired_direction_rate,paired_conditions,mean_distinct_answers_per_repeated_condition,semantic_support,causal_claim_review,full_interpretation_change,semantic_repeat_consistency
0,Haiku 4.5,3,0,0,0,0,3,None,None,None,...,1.0,1.0,1.0,None,0,1.0,None,None,None,None
1,Opus 4.6,2,1,0,0,0,2,None,None,None,...,1.0,1.0,1.0,None,0,NaN,None,None,None,None
2,Sonnet 4.6,1,0,0,0,0,1,None,None,None,...,1.0,1.0,1.0,None,0,NaN,None,None,None,None


Paid experiment folder: c:\Users\katyng\Downloads\AI_Agents\rag-development\reports\runs\numerical_evidence\20261020T203137237244Z-c1b7697d
